# 01 — Data Exploration

Schema audit and quality checks on the WRDS / OptionMetrics SPY parquet, the VIX / VXN feeds, and the Treasury rate file. The goal is to confirm coverage, surface duplicate-key issues, and document the type-coercion rules used downstream.

## Setup

In [ ]:
from pathlib import Path
import polars as pl, pandas as pd, numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

DATA = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. Concatenate yearly SPY option files

In [ ]:
files = sorted((DATA / 'spy').glob('spy_options_*.parquet'))
print(f'{len(files)} yearly files')
df = pl.concat([pl.read_parquet(f) for f in files], how='vertical')
print('total rows:', f'{df.height:,}')
print('columns:', df.columns)

## 2. Schema and date coverage

In [ ]:
df_pd = df.head(50_000).to_pandas()
df_pd['date'] = pd.to_datetime(df_pd['date'])
df_pd['exdate'] = pd.to_datetime(df_pd['exdate'])
print('date range :', df_pd['date'].min(), '..', df_pd['date'].max())
print('cp_flag    :', df_pd['cp_flag'].unique())
print('strike     :', df_pd['strike_price'].describe())
# strike_price is in $ × 1000 (a 400-strike is stored as 400000)

## 3. Duplicate-key audit

SPY has weekly + monthly contracts at the same `(date, expir, strike)`. We dedupe by keeping the row with the highest combined OI (call + put).

In [ ]:
dup = (df.group_by(['date','exdate','strike_price','cp_flag']).len()
         .filter(pl.col('len') > 1).head(10).to_pandas())
print(f'duplicate rows count: {dup.shape[0]:,}')
dup

## 4. Missing-IV / quality flags

In [ ]:
samp = df.head(1_000_000).to_pandas()
print('finite IV%        :', samp.impl_volatility.notna().mean())
print('positive bid%     :', (samp.best_bid > 0).mean())
print('finite delta%     :', samp.delta.notna().mean())
print('zero OI rows%     :', (samp.open_interest == 0).mean())

## 5. Free macro feeds — VIX / Treasury

Load VIX/VXN from yfinance and Treasury from FRED, sanity-check coverage.

In [ ]:
import yfinance as yf, warnings
warnings.filterwarnings('ignore')
vix = yf.Ticker('^VIX').history(period='max')['Close']
vxn = yf.Ticker('^VXN').history(period='max')['Close']
print('VIX:', len(vix), vix.index[0].date(), '..', vix.index[-1].date())
print('VXN:', len(vxn), vxn.index[0].date(), '..', vxn.index[-1].date())
spread = (vxn - vix).rename('VXN - VIX').dropna()
spread.tail(60).plot(figsize=(11,3), title='VXN - VIX (last 60 days)')

## 6. Conclusions for the pipeline

* Strike must be divided by 1000.
* Multi-root duplicates affect ~3% of rows; dedupe by max-OI before further work.
* Greeks from OptionMetrics are computed against their proprietary IV; we use them directly.
* IV-finite + positive-bid rules drop <0.01% of rows.
* VIX/VXN are clean through to today via yfinance.
* The actual transform/dedup is implemented in `src.data_pipeline`.